# Module 4.1: ReAct Agents - Reasoning and Acting

## Learning Objectives

By completing this notebook, you will:
1. Understand the ReAct (Reasoning + Acting) pattern for agent reasoning
2. Implement a ReAct agent loop from scratch
3. Parse and execute Thought-Action-Observation cycles
4. Handle multi-step reasoning tasks that require planning
5. Debug reasoning failures and improve agent prompts

## Prerequisites

- Module 2 completed (tool use and multi-tool agents)
- Understanding of agent loops
- Familiarity with prompt engineering

## Estimated Time: 75 minutes

---

## 1. Setup & Imports

We'll build a ReAct agent that explicitly shows its reasoning process.

In [ ]:
# Install required packages
# !pip install anthropic openai

import json
import os
import re
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, field
from enum import Enum
import anthropic

print("✓ Imports successful")

## 2. Understanding ReAct

**ReAct** = **Rea**soning + **Act**ing

Instead of directly calling tools, ReAct agents:
1. **Think**: Reason about what to do next
2. **Act**: Choose and execute an action (tool call)
3. **Observe**: See the result of the action
4. **Repeat**: Continue until the task is complete

**Example ReAct trace**:
```
Question: What is the capital of the country that won the 2022 FIFA World Cup?

Thought 1: I need to find out which country won the 2022 World Cup first.
Action 1: search("2022 FIFA World Cup winner")
Observation 1: Argentina won the 2022 FIFA World Cup.

Thought 2: Now I know Argentina won. I need to find Argentina's capital.
Action 2: search("capital of Argentina")
Observation 2: Buenos Aires is the capital of Argentina.

Thought 3: I have the answer.
Final Answer: Buenos Aires
```

**Why ReAct matters**:
- Makes reasoning transparent and debuggable
- Enables multi-step problem solving
- Allows verification of intermediate steps
- Better handles complex tasks than direct tool calling

## 3. ReAct Step Types

We'll define the types of steps a ReAct agent can take.

In [ ]:
# Purpose: Define structured types for ReAct steps
# Key Concept: ReAct uses explicit reasoning steps before actions

class StepType(Enum):
    """Types of steps in ReAct reasoning."""
    THOUGHT = "thought"
    ACTION = "action"
    OBSERVATION = "observation"
    FINAL_ANSWER = "final_answer"

@dataclass
class ReActStep:
    """A single step in ReAct reasoning."""
    step_type: StepType
    content: str
    step_number: int
    action_name: Optional[str] = None
    action_input: Optional[Dict[str, Any]] = None
    
    def __str__(self) -> str:
        if self.step_type == StepType.THOUGHT:
            return f"Thought {self.step_number}: {self.content}"
        elif self.step_type == StepType.ACTION:
            return f"Action {self.step_number}: {self.action_name}({json.dumps(self.action_input)})"
        elif self.step_type == StepType.OBSERVATION:
            return f"Observation {self.step_number}: {self.content}"
        else:
            return f"Final Answer: {self.content}"

@dataclass
class ReActTrace:
    """Complete ReAct reasoning trace."""
    question: str
    steps: List[ReActStep] = field(default_factory=list)
    final_answer: Optional[str] = None
    
    def add_step(self, step: ReActStep) -> None:
        """Add a step to the trace."""
        self.steps.append(step)
    
    def display(self) -> str:
        """Generate formatted display of reasoning trace."""
        output = f"Question: {self.question}\n\n"
        for step in self.steps:
            output += str(step) + "\n"
        if self.final_answer:
            output += f"\nFinal Answer: {self.final_answer}"
        return output

# Example trace
example_trace = ReActTrace(question="What is 5 + 3?")
example_trace.add_step(ReActStep(
    step_type=StepType.THOUGHT,
    content="I need to add 5 and 3",
    step_number=1
))
example_trace.add_step(ReActStep(
    step_type=StepType.ACTION,
    content="calculator",
    step_number=1,
    action_name="calculator",
    action_input={"operation": "add", "a": 5, "b": 3}
))
example_trace.add_step(ReActStep(
    step_type=StepType.OBSERVATION,
    content="8",
    step_number=1
))
example_trace.final_answer = "8"

print(example_trace.display())

## 4. ReAct Prompting

The key to ReAct is the **prompt structure**. We need to instruct the LLM to:
- Think before acting
- Use specific format for thoughts and actions
- Know when to give a final answer

In [ ]:
# Purpose: Create ReAct system prompt
# Key Concept: Prompt structure shapes agent behavior

def create_react_prompt(available_tools: List[str]) -> str:
    """
    Create system prompt for ReAct agent.
    
    Parameters
    ----------
    available_tools : list of str
        Names of available tools
    
    Returns
    -------
    str
        System prompt
    """
    tools_list = "\n".join([f"- {tool}" for tool in available_tools])
    
    prompt = f"""
You are a ReAct (Reasoning + Acting) agent. You solve problems by alternating between Thought and Action steps.

AVAILABLE TOOLS:
{tools_list}

FORMAT:
For each reasoning cycle, output EXACTLY this format:

Thought: [Your reasoning about what to do next]
Action: tool_name
Action Input: {{"param": "value"}}

After you see the Observation (tool result), continue with the next Thought-Action cycle.

When you have enough information to answer the question, output:
Thought: [Your reasoning about why you can answer now]
Final Answer: [Your answer to the original question]

RULES:
1. Always start with a Thought
2. Only use available tools
3. Action Input must be valid JSON
4. Give Final Answer when you have sufficient information
5. Keep thoughts concise and focused

EXAMPLE:
Question: What is 15% of 200?

Thought: I need to calculate 15% of 200.
Action: percentage
Action Input: {{"value": 200, "percentage": 15}}
Observation: {{"success": true, "result": 30}}

Thought: I have the answer from the calculation.
Final Answer: 15% of 200 is 30.
"""
    return prompt.strip()

# Example prompt
example_prompt = create_react_prompt(["calculator", "search", "store", "retrieve"])
print(example_prompt[:500] + "...")

## 5. ReAct Parser

We need to parse the LLM's ReAct output to extract:
- Thoughts
- Actions (tool name + input)
- Final answers

In [ ]:
# Purpose: Parse ReAct formatted output from LLM
# Key Concept: Structured parsing enables reliable agent loops

class ReActParser:
    """Parse ReAct formatted text from LLM."""
    
    @staticmethod
    def parse_step(text: str, step_number: int) -> Optional[Dict[str, Any]]:
        """
        Parse a ReAct step from text.
        
        Parameters
        ----------
        text : str
            LLM output text
        step_number : int
            Current step number
        
        Returns
        -------
        dict or None
            Parsed step information or None if parsing failed
        """
        # Check for Final Answer
        final_pattern = r'Final Answer:\s*(.+?)(?:\n|$)'
        final_match = re.search(final_pattern, text, re.IGNORECASE | re.DOTALL)
        if final_match:
            return {
                "type": "final_answer",
                "content": final_match.group(1).strip()
            }
        
        # Parse Thought
        thought_pattern = r'Thought:\s*(.+?)(?:\n|$)'
        thought_match = re.search(thought_pattern, text, re.IGNORECASE)
        
        # Parse Action
        action_pattern = r'Action:\s*([\w_]+)'
        action_match = re.search(action_pattern, text, re.IGNORECASE)
        
        # Parse Action Input
        input_pattern = r'Action Input:\s*(\{.+?\})'
        input_match = re.search(input_pattern, text, re.IGNORECASE | re.DOTALL)
        
        if thought_match and action_match and input_match:
            try:
                action_input = json.loads(input_match.group(1))
                return {
                    "type": "action",
                    "thought": thought_match.group(1).strip(),
                    "action_name": action_match.group(1).strip(),
                    "action_input": action_input,
                    "step_number": step_number
                }
            except json.JSONDecodeError:
                return None
        
        elif thought_match:
            # Just a thought, no action yet
            return {
                "type": "thought_only",
                "thought": thought_match.group(1).strip(),
                "step_number": step_number
            }
        
        return None

# Test parser
test_text = """
Thought: I need to calculate 20% of 500.
Action: percentage
Action Input: {"value": 500, "percentage": 20}
"""

parsed = ReActParser.parse_step(test_text, 1)
print("Parsed:", json.dumps(parsed, indent=2))

## 6. ReAct Agent Implementation

Now we'll build a complete ReAct agent that uses the prompting and parsing we've defined.

In [ ]:
# Purpose: Implement complete ReAct agent
# Key Concept: ReAct loop alternates between reasoning and tool execution

class ReActAgent:
    """ReAct (Reasoning + Acting) agent implementation."""
    
    def __init__(self, api_key: Optional[str] = None, verbose: bool = True):
        """
        Initialize ReAct agent.
        
        Parameters
        ----------
        api_key : str, optional
            Anthropic API key
        verbose : bool
            Print reasoning steps
        """
        self.client = anthropic.Anthropic(
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY")
        )
        self.tools = {}
        self.verbose = verbose
        self.parser = ReActParser()
    
    def register_tool(self, name: str, implementation: callable, description: str) -> None:
        """Register a tool."""
        self.tools[name] = {
            "implementation": implementation,
            "description": description
        }
        if self.verbose:
            print(f"✓ Registered: {name}")
    
    def execute_action(self, action_name: str, action_input: Dict) -> str:
        """
        Execute a tool action.
        
        Parameters
        ----------
        action_name : str
            Tool name
        action_input : dict
            Tool parameters
        
        Returns
        -------
        str
            Observation (tool result as string)
        """
        if action_name not in self.tools:
            return f"Error: Unknown tool '{action_name}'"
        
        try:
            result = self.tools[action_name]["implementation"](**action_input)
            return json.dumps(result)
        except Exception as e:
            return f"Error executing {action_name}: {str(e)}"
    
    def run(self, question: str, max_iterations: int = 10) -> Tuple[str, ReActTrace]:
        """
        Run ReAct agent on a question.
        
        Parameters
        ----------
        question : str
            Question to answer
        max_iterations : int
            Maximum reasoning iterations
        
        Returns
        -------
        tuple
            (final_answer, trace)
        """
        # Initialize
        trace = ReActTrace(question=question)
        system_prompt = create_react_prompt(list(self.tools.keys()))
        conversation_history = f"Question: {question}\n\n"
        
        step_number = 1
        
        for iteration in range(max_iterations):
            if self.verbose:
                print(f"\n{'='*60}")
                print(f"Iteration {iteration + 1}")
                print("="*60)
            
            # Get LLM response
            response = self.client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                system=system_prompt,
                messages=[{"role": "user", "content": conversation_history}]
            )
            
            response_text = response.content[0].text
            
            if self.verbose:
                print(f"\nLLM Output:\n{response_text}")
            
            # Parse the step
            parsed = self.parser.parse_step(response_text, step_number)
            
            if not parsed:
                return "Error: Could not parse LLM output", trace
            
            # Handle different step types
            if parsed["type"] == "final_answer":
                # We have the final answer
                final_answer = parsed["content"]
                trace.final_answer = final_answer
                
                if self.verbose:
                    print(f"\n✓ Final Answer: {final_answer}")
                
                return final_answer, trace
            
            elif parsed["type"] == "action":
                # Record thought
                trace.add_step(ReActStep(
                    step_type=StepType.THOUGHT,
                    content=parsed["thought"],
                    step_number=step_number
                ))
                
                # Record action
                trace.add_step(ReActStep(
                    step_type=StepType.ACTION,
                    content=parsed["action_name"],
                    step_number=step_number,
                    action_name=parsed["action_name"],
                    action_input=parsed["action_input"]
                ))
                
                # Execute action
                observation = self.execute_action(
                    parsed["action_name"],
                    parsed["action_input"]
                )
                
                # Record observation
                trace.add_step(ReActStep(
                    step_type=StepType.OBSERVATION,
                    content=observation,
                    step_number=step_number
                ))
                
                if self.verbose:
                    print(f"\nObservation: {observation}")
                
                # Update conversation history
                conversation_history += response_text + "\n"
                conversation_history += f"Observation: {observation}\n\n"
                
                step_number += 1
            
            else:
                # Just a thought, continue
                conversation_history += response_text + "\n\n"
        
        return "Max iterations reached without final answer", trace

print("ReActAgent class defined")

## 7. Testing ReAct Agent

Let's create an agent with tools and test it on multi-step reasoning tasks.

In [ ]:
# Purpose: Test ReAct agent on multi-step task
# Key Concept: ReAct shines on tasks requiring sequential reasoning

# Create simple tools
def calculator(operation: str, a: float, b: float) -> Dict:
    """Perform basic arithmetic."""
    ops = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else None
    }
    result = ops.get(operation)
    if result is None:
        return {"success": False, "error": "Invalid operation or division by zero"}
    return {"success": True, "result": result}

# Mock search
def search(query: str) -> Dict:
    """Mock search tool."""
    # Simulated knowledge base
    knowledge = {
        "argentina population": "Argentina has a population of approximately 45 million people.",
        "tokyo population": "Tokyo has a population of approximately 14 million people.",
        "france capital": "The capital of France is Paris."
    }
    
    for key, value in knowledge.items():
        if key in query.lower():
            return {"success": True, "result": value}
    
    return {"success": False, "error": f"No information found for: {query}"}

# Create agent
agent = ReActAgent(verbose=True)
agent.register_tool("calculator", calculator, "Perform arithmetic operations")
agent.register_tool("search", search, "Search for information")

print("\nAgent ready with tools: calculator, search")

In [ ]:
# Test multi-step reasoning
print("\n" + "#"*60)
print("TEST: Multi-Step Math Problem")
print("#"*60)

question = "Calculate 25 * 4, then add 100 to that result."
answer, trace = agent.run(question)

print("\n" + "="*60)
print("COMPLETE TRACE")
print("="*60)
print(trace.display())

## Hands-On Exercises

Build and test your own ReAct agents.

### Exercise 4.1.1: Improve the Parser

**Task**: Enhance the ReActParser to handle variations in formatting (e.g., "Thought:" vs "Thought 1:" vs "THOUGHT:").

**Expected Output**: Parser that's more robust to formatting variations.

**Hints**:
<details>
<summary>Hint 1</summary>
Use case-insensitive regex: re.IGNORECASE flag.
</details>

<details>
<summary>Hint 2</summary>
Make the colon and step number optional in the pattern: r'Thought\s*\d*\s*:?\s*(.+)'.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def improved_parse_thought(text: str) -> Optional[str]:
    """
    Parse thought with flexible formatting.
    
    Should handle:
    - "Thought: ..."
    - "Thought 1: ..."
    - "THOUGHT: ..."
    - "thought ..."
    
    Parameters
    ----------
    text : str
        Text to parse
    
    Returns
    -------
    str or None
        Extracted thought or None
    """
    # TODO: Implement flexible thought parsing
    pass

exercise_4_1_1_answer = improved_parse_thought  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_4_1_1():
    test_cases = [
        "Thought: I need to calculate this",
        "Thought 1: I need to calculate this",
        "THOUGHT: I need to calculate this",
        "thought I need to calculate this"
    ]
    
    for test in test_cases:
        result = exercise_4_1_1_answer(test)
        assert result is not None, f"Failed to parse: {test}"
        assert "calculate" in result.lower(), f"Incorrect extraction from: {test}"
    
    print("✓ Exercise 4.1.1 passed!")

test_exercise_4_1_1()

### Exercise 4.1.2: Add Memory Tool

**Task**: Create a memory tool that the ReAct agent can use to store intermediate results.

**Expected Output**: Tool function that stores and retrieves values.

**Hints**:
<details>
<summary>Hint 1</summary>
Use a dictionary to store key-value pairs.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

_memory = {}

def memory_tool(action: str, key: str, value: Optional[str] = None) -> Dict:
    """
    Store or retrieve from memory.
    
    Parameters
    ----------
    action : str
        Either 'store' or 'retrieve'
    key : str
        Memory key
    value : str, optional
        Value to store (required if action='store')
    
    Returns
    -------
    dict
        Result with success status
    """
    # TODO: Implement memory tool
    pass

exercise_4_1_2_answer = memory_tool  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_4_1_2():
    # Test store
    result = exercise_4_1_2_answer("store", "test_key", "test_value")
    assert isinstance(result, dict), "Should return dict"
    assert result.get("success") == True, "Store should succeed"
    
    # Test retrieve
    result = exercise_4_1_2_answer("retrieve", "test_key")
    assert result.get("success") == True, "Retrieve should succeed"
    assert "value" in result or "result" in result, "Should return stored value"
    
    print("✓ Exercise 4.1.2 passed!")

test_exercise_4_1_2()

### Exercise 4.1.3: Trace Analyzer

**Task**: Write a function that analyzes a ReActTrace and returns metrics.

**Expected Output**: Dict with: total_steps, total_actions, total_thoughts, action_types_used.

**Hints**:
<details>
<summary>Hint 1</summary>
Iterate through trace.steps and count by step_type.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def analyze_react_trace(trace: ReActTrace) -> Dict[str, Any]:
    """
    Analyze ReAct trace.
    
    Parameters
    ----------
    trace : ReActTrace
        Trace to analyze
    
    Returns
    -------
    dict
        Analysis metrics
    """
    # TODO: Analyze trace and return metrics
    pass

exercise_4_1_3_answer = analyze_react_trace  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_4_1_3():
    # Create test trace
    test_trace = ReActTrace("test question")
    test_trace.add_step(ReActStep(StepType.THOUGHT, "thinking", 1))
    test_trace.add_step(ReActStep(StepType.ACTION, "action", 1, action_name="calculator"))
    test_trace.add_step(ReActStep(StepType.OBSERVATION, "result", 1))
    test_trace.add_step(ReActStep(StepType.THOUGHT, "more thinking", 2))
    
    result = exercise_4_1_3_answer(test_trace)
    
    assert isinstance(result, dict), "Should return dict"
    assert "total_steps" in result, "Should count total steps"
    assert result["total_steps"] == 4, "Should count 4 steps"
    assert "total_actions" in result or "action_count" in result, "Should count actions"
    assert "total_thoughts" in result or "thought_count" in result, "Should count thoughts"
    
    print("✓ Exercise 4.1.3 passed!")

test_exercise_4_1_3()

## Solutions

In [ ]:
# SOLUTION 4.1.1: Improved Parser

def improved_parse_thought_solution(text: str) -> Optional[str]:
    pattern = r'thought\s*\d*\s*:?\s*(.+?)(?:\n|$)'
    match = re.search(pattern, text, re.IGNORECASE)
    return match.group(1).strip() if match else None

# SOLUTION 4.1.2: Memory Tool

def memory_tool_solution(action: str, key: str, value: Optional[str] = None) -> Dict:
    if action == "store":
        if value is None:
            return {"success": False, "error": "Value required for store"}
        _memory[key] = value
        return {"success": True, "message": f"Stored '{value}' at '{key}'"}
    
    elif action == "retrieve":
        if key in _memory:
            return {"success": True, "value": _memory[key]}
        return {"success": False, "error": f"Key '{key}' not found"}
    
    else:
        return {"success": False, "error": f"Unknown action: {action}"}

# SOLUTION 4.1.3: Trace Analyzer

def analyze_react_trace_solution(trace: ReActTrace) -> Dict[str, Any]:
    from collections import Counter
    
    thoughts = [s for s in trace.steps if s.step_type == StepType.THOUGHT]
    actions = [s for s in trace.steps if s.step_type == StepType.ACTION]
    observations = [s for s in trace.steps if s.step_type == StepType.OBSERVATION]
    
    action_names = [a.action_name for a in actions if a.action_name]
    action_counts = Counter(action_names)
    
    return {
        "total_steps": len(trace.steps),
        "total_thoughts": len(thoughts),
        "total_actions": len(actions),
        "total_observations": len(observations),
        "action_types_used": dict(action_counts),
        "unique_actions": len(action_counts)
    }

## Summary

**Key Takeaways**:

1. **ReAct Pattern**: Explicit reasoning before action improves task completion
2. **Structured Output**: Formatting guides enable reliable parsing
3. **Transparency**: Thought traces make debugging and verification possible
4. **Multi-Step Reasoning**: ReAct excels at tasks requiring sequential planning
5. **Prompt Engineering**: System prompt structure critically impacts agent behavior

**What's Next**:
- Module 4.2: Planning agents with goal decomposition
- Module 4.3: Self-reflection and error correction
- Module 5: Memory and state management

**Additional Resources**:
- [ReAct Paper](https://arxiv.org/abs/2210.03629) - Original research
- [LangChain ReAct Agent](https://python.langchain.com/docs/modules/agents/agent_types/react)

---

You've built a ReAct agent from scratch! This pattern is foundational for advanced agentic systems.